<a href="https://colab.research.google.com/github/paridhietal/TransferLearning/blob/main/transferLearning1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-genai --quiet

In [ ]:
import os
import json
import uuid
import datetime
from google import genai
from google.genai import types
from google.colab import userdata
userdata.get('GROQ_API_KEY')

# No Drive — local storage only
MEMORY_PATH = "/content/task_memory.json"

if not os.path.exists(MEMORY_PATH):
    with open(MEMORY_PATH, "w") as f:
        json.dump([], f)
    print("Created fresh memory file.")
else:
    with open(MEMORY_PATH, "r") as f:
        existing = json.load(f)
    print(f"Found existing memory with {len(existing)} task(s).")

client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client ready! (google-genai new SDK)")

Created fresh memory file.
Gemini client ready! (google-genai new SDK)


In [ ]:

# ── CELL 4: Core memory functions ────────────────────────────

def load_memory():
    """Load all past tasks from the JSON file."""
    with open(MEMORY_PATH, "r") as f:
        return json.load(f)


def save_memory(memory: list):
    """Save the full memory list back to the JSON file."""
    with open(MEMORY_PATH, "w") as f:
        json.dump(memory, f, indent=2)


def log_task(task: str, output: str, score: float, lesson: str = ""):
    """
    Save a completed task to memory.

    Args:
        task:   The input prompt / task description
        output: The AI's response
        score:  Quality score between 0.0 (bad) and 1.0 (great)
        lesson: Optional note on what worked or what to do differently
    """
    memory = load_memory()

    entry = {
        "id":        str(uuid.uuid4())[:8],
        "timestamp": datetime.datetime.now().isoformat(),
        "task":      task,
        "output":    output,
        "score":     score,
        "lesson":    lesson,
    }

    memory.append(entry)
    save_memory(memory)
    print(f"[Memory] Saved task #{len(memory)} (score={score})")
    return entry


def get_relevant_examples(current_task: str, top_k: int = 3) -> list:
    """
    Retrieve the top_k most relevant past tasks to use as examples.

    Uses keyword overlap for now — Notebook 2 replaces this
    with proper vector search using ChromaDB + embeddings.

    Args:
        current_task: The new task we want help with
        top_k:        How many past examples to retrieve

    Returns:
        List of past task entries, sorted by relevance
    """
    memory = load_memory()

    if not memory:
        return []

    current_words = set(current_task.lower().split())

    scored = []
    for entry in memory:
        past_words  = set(entry["task"].lower().split())
        overlap     = len(current_words & past_words)   # word intersection
        quality_bonus = entry["score"]                  # prefer high-quality memories
        relevance   = overlap + quality_bonus
        scored.append((relevance, entry))

    scored.sort(key=lambda x: x[0], reverse=True)
    top_entries = [entry for _, entry in scored[:top_k]]

    print(f"[Memory] Found {len(top_entries)} relevant past example(s)")
    return top_entries


def build_prompt_with_memory(task: str, examples: list) -> str:
    """
    Build a full prompt that injects past memories as context.

    With Gemini we pass everything as a single combined prompt
    (system context + user task together).

    Args:
        task:     The current task
        examples: Past task entries from get_relevant_examples()

    Returns:
        Full prompt string ready to send to Gemini
    """
    prompt = "You are a helpful AI assistant that learns from experience.\n\n"

    if examples:
        prompt += "=== PAST EXPERIENCE ===\n"
        prompt += "Here are similar tasks you've handled before. "
        prompt += "Use these lessons to improve your response:\n\n"

        for i, ex in enumerate(examples, 1):
            prompt += f"Example {i} (quality score: {ex['score']}):\n"
            prompt += f"  Task:   {ex['task']}\n"
            prompt += f"  Output: {ex['output'][:300]}"
            prompt += "...\n" if len(ex['output']) > 300 else "\n"
            if ex.get("lesson"):
                prompt += f"  Lesson: {ex['lesson']}\n"
            prompt += "\n"

        prompt += "=== END OF PAST EXPERIENCE ===\n\n"
        prompt += "Now apply those lessons to this new task:\n\n"
    else:
        prompt += "This is your first task — no past experience yet.\n\n"
        prompt += "Current task:\n\n"

    prompt += task
    return prompt


print("Memory functions loaded!")

Memory functions loaded!


In [ ]:
# ── CELL 5: The main agent function ──────────────────────────

def run_task(task: str) -> str:
    """
    Run a task using Gemini, with memory context injected.

    Steps:
      1. Retrieve relevant past examples from memory
      2. Build a memory-enriched prompt
      3. Call Gemini API
      4. Return the output

    Args:
        task: The task description / user prompt

    Returns:
        Gemini's response string
    """
    print(f"\n{'='*55}")
    print(f"Task: {task}")
    print(f"{'='*55}")

    # Step 1: retrieve relevant memories
    examples = get_relevant_examples(task, top_k=3)

    # Step 2: build memory-injected prompt
    full_prompt = build_prompt_with_memory(task, examples)

    # Step 3: call Gemini
    def run_task(task: str) -> str:
     print(f"\n{'='*55}")
    print(f"Task: {task}")
    print(f"{'='*55}")

    examples = get_relevant_examples(task, top_k=3)
    full_prompt = build_prompt_with_memory(task, examples)

    # New SDK syntax
    response = client.models.generate_content(
    model="gemini-2.5-flash",
        contents=full_prompt,
        config=types.GenerateContentConfig(
            temperature=0.7,
            max_output_tokens=600,
        )
    )

    output = response.text
    print(f"\nOutput:\n{output}")
    return output

    output = response.text
    print(f"\nOutput:\n{output}")
    return output


print("Agent function ready!")

Agent function ready!


In [ ]:
 # ── CELL 6: Run your first task (no memory yet) ──────────────

task1 = "Write a Python function that reverses a string"
output1 = run_task(task1)


Task: Write a Python function that reverses a string
Task: Write a Python function that reverses a string

Output:
Okay, this sounds like a great first task! I'm ready to learn and help.

Here's a Python function that reverses a string, along with an explanation and an example:

```python
def reverse_string(s: str) -> str:
    """
    Reverses a given string.

    This function takes a string as input and returns a new string
    with the characters in reverse order. It uses Python's string slicing
    feature to achieve this efficiently and concisely.

    Args:
        s (str): The input string to be reversed.

    Returns:
        str: The reversed string.

    Example:
        >>> reverse_string("hello")
        "olleh"
        >>> reverse_string("Python")
        "nohtyP"
        >>> reverse_string("")
        ""
    """
    return s[::-1]

# --- Example Usage ---
if __name__ == "__main__":
    my_string_1 = "Hello, World!"
    reversed_string_1 = reverse_string(my_string_1)
    

In [ ]:
# Run this as a separate cell to list all available models
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/deep-research-prev

In [ ]:

# ── CELL 7: Score and log Task 1 ─────────────────────────────
# Read the output above, decide how good it was (0.0 to 1.0),
# then write a lesson for future runs.

log_task(
    task    = task1,
    output  = output1,
    score   = 0.85,    # <-- change this based on how good the output was
    lesson  = "Good clean solution. Include a docstring and example usage next time."
)

[Memory] Saved task #1 (score=0.85)


{'id': 'effe21a4',
 'timestamp': '2026-05-11T18:13:54.363875',
 'task': 'Write a Python function that reverses a string',
 'output': 'Okay, this sounds like a great first task! I\'m ready to learn and help.\n\nHere\'s a Python function that reverses a string, along with an explanation and an example:\n\n```python\ndef reverse_string(s: str) -> str:\n    """\n    Reverses a given string.\n\n    This function takes a string as input and returns a new string\n    with the characters in reverse order. It uses Python\'s string slicing\n    feature to achieve this efficiently and concisely.\n\n    Args:\n        s (str): The input string to be reversed.\n\n    Returns:\n        str: The reversed string.\n\n    Example:\n        >>> reverse_string("hello")\n        "olleh"\n        >>> reverse_string("Python")\n        "nohtyP"\n        >>> reverse_string("")\n        ""\n    """\n    return s[::-1]\n\n# --- Example Usage ---\nif __name__ == "__main__":\n    my_string_1 = "Hello, World!"\n   

In [ ]:
# ── CELL 8: Run a second task (memory kicks in) ──────────────
# Similar to Task 1 — Gemini will find it in memory and apply
# the lesson from the previous task automatically.

task2 = "Write a Python function that checks if a string is a palindrome"
output2 = run_task(task2)


Task: Write a Python function that checks if a string is a palindrome
[Memory] Found 1 relevant past example(s)
Task: Write a Python function that checks if a string is a palindrome
[Memory] Found 1 relevant past example(s)

Output:
Okay, this sounds like a great task! I'm ready to learn and help.

Here's a Python function that checks if a string is a palindrome, along with an explanation, a docstring, and example usage:

```python
import re

def is_palindrome(s: str) -> bool:
    """
    Checks if a given string is a palindrome.

    A palindrome is a word, phrase, number, or other sequence of characters
    which reads the same backward as forward, ignoring spaces, punctuation,
    and capitalization.

    Args:
        s (str): The input string to check.

    Returns:
        bool: True if the string is a palindrome, False otherwise.
    """
    # Step 1: Normalize the string
    # Convert to lowercase to handle case-insensitivity (e.g., "Racecar")
    # Remove non-alphanumeric cha

In [ ]:
# ── CELL 9: Score and log Task 2 ─────────────────────────────

log_task(
    task    = task2,
    output  = output2,
    score   = 0.90,
    lesson  = "Handle edge cases: ignore spaces, punctuation, and capitalization."
)


[Memory] Saved task #2 (score=0.9)


{'id': 'fb6419dd',
 'timestamp': '2026-05-11T18:13:58.066516',
 'task': 'Write a Python function that checks if a string is a palindrome',
 'output': 'Okay, this sounds like a great task! I\'m ready to learn and help.\n\nHere\'s a Python function that checks if a string is a palindrome, along with an explanation, a docstring, and example usage:\n\n```python\nimport re\n\ndef is_palindrome(s: str) -> bool:\n    """\n    Checks if a given string is a palindrome.\n\n    A palindrome is a word, phrase, number, or other sequence of characters\n    which reads the same backward as forward, ignoring spaces, punctuation,\n    and capitalization.\n\n    Args:\n        s (str): The input string to check.\n\n    Returns:\n        bool: True if the string is a palindrome, False otherwise.\n    """\n    # Step 1: Normalize the string\n    # Convert to lowercase to handle case-insensitivity (e.g., "Racecar")\n    # Remove non-alphanumeric characters using a regular expression\n    # This ensures "A 

In [ ]:

# ── CELL 10: Run a different kind of task ────────────────────

task3 = "Explain gradient descent in simple terms for a beginner"
output3 = run_task(task3)

log_task(
    task    = task3,
    output  = output3,
    score   = 0.80,
    lesson  = "Good analogy used. Next time add a small numeric example to make it concrete."
)


Task: Explain gradient descent in simple terms for a beginner
[Memory] Found 2 relevant past example(s)
Task: Explain gradient descent in simple terms for a beginner
[Memory] Found 2 relevant past example(s)

Output:
Okay, this sounds like a great task! I'm ready to learn and help.

Gradient Descent is a fundamental optimization algorithm used in machine learning to find the "best fit" for a model. Let's break it down in simple terms using an analogy:

---

### Gradient Descent: Finding the Lowest Point

Imagine you are a **blindfolded hiker** stuck on a mountain, and your goal is to reach the **lowest point** (the valley floor) as quickly as possible. You can't see the whole mountain, but you can feel the slope right where you're standing.

Here's how you'd likely try to get down:

1.  **Feel the Slope (Calculate the Gradient):
[Memory] Saved task #3 (score=0.8)


{'id': 'c75fa3eb',
 'timestamp': '2026-05-11T18:14:01.663996',
 'task': 'Explain gradient descent in simple terms for a beginner',
 'output': 'Okay, this sounds like a great task! I\'m ready to learn and help.\n\nGradient Descent is a fundamental optimization algorithm used in machine learning to find the "best fit" for a model. Let\'s break it down in simple terms using an analogy:\n\n---\n\n### Gradient Descent: Finding the Lowest Point\n\nImagine you are a **blindfolded hiker** stuck on a mountain, and your goal is to reach the **lowest point** (the valley floor) as quickly as possible. You can\'t see the whole mountain, but you can feel the slope right where you\'re standing.\n\nHere\'s how you\'d likely try to get down:\n\n1.  **Feel the Slope (Calculate the Gradient):',
 'score': 0.8,
 'lesson': 'Good analogy used. Next time add a small numeric example to make it concrete.'}

In [ ]:
# ── CELL 11: Inspect your memory store ───────────────────────

def show_memory():
    """Print a readable summary of everything in memory."""
    memory = load_memory()

    if not memory:
        print("Memory is empty — run some tasks first!")
        return

    print(f"\nMemory store — {len(memory)} task(s) logged\n")
    print("=" * 60)

    for entry in memory:
        print(f"ID:      {entry['id']}")
        print(f"Time:    {entry['timestamp'][:19]}")
        print(f"Score:   {entry['score']}")
        print(f"Task:    {entry['task']}")
        print(f"Lesson:  {entry.get('lesson', 'none')}")
        print("-" * 60)

show_memory()


Memory store — 3 task(s) logged

ID:      effe21a4
Time:    2026-05-11T18:13:54
Score:   0.85
Task:    Write a Python function that reverses a string
Lesson:  Good clean solution. Include a docstring and example usage next time.
------------------------------------------------------------
ID:      fb6419dd
Time:    2026-05-11T18:13:58
Score:   0.9
Task:    Write a Python function that checks if a string is a palindrome
Lesson:  Handle edge cases: ignore spaces, punctuation, and capitalization.
------------------------------------------------------------
ID:      c75fa3eb
Time:    2026-05-11T18:14:01
Score:   0.8
Task:    Explain gradient descent in simple terms for a beginner
Lesson:  Good analogy used. Next time add a small numeric example to make it concrete.
------------------------------------------------------------


In [ ]:
# ── CELL 12: Test retrieval on a new task ────────────────────
# Watch which past memories get pulled for a brand new task.

new_task = "Write a Python function to count vowels in a string"

print("Retrieving relevant memories for:")
print(f"  '{new_task}'\n")

examples = get_relevant_examples(new_task, top_k=3)

print("\nTop retrieved memories:")
for i, ex in enumerate(examples, 1):
    print(f"\n  [{i}] Score={ex['score']} | Task: {ex['task']}")
    print(f"       Lesson: {ex.get('lesson', 'none')}")

# Now run it with memory injected automatically
output_new = run_task(new_task)

log_task(
    task    = new_task,
    output  = output_new,
    score   = 0.88,
    lesson  = "Correct solution. Add type hints and handle uppercase vowels explicitly."
)

Retrieving relevant memories for:
  'Write a Python function to count vowels in a string'

[Memory] Found 3 relevant past example(s)

Top retrieved memories:

  [1] Score=0.9 | Task: Write a Python function that checks if a string is a palindrome
       Lesson: Handle edge cases: ignore spaces, punctuation, and capitalization.

  [2] Score=0.85 | Task: Write a Python function that reverses a string
       Lesson: Good clean solution. Include a docstring and example usage next time.

  [3] Score=0.8 | Task: Explain gradient descent in simple terms for a beginner
       Lesson: Good analogy used. Next time add a small numeric example to make it concrete.

Task: Write a Python function to count vowels in a string
[Memory] Found 3 relevant past example(s)
Task: Write a Python function to count vowels in a string
[Memory] Found 3 relevant past example(s)

Output:
Okay, this sounds like a great task! I'm ready to learn and help.

Here's a Python function that counts vowels in a string, incor

{'id': 'be4c4d5e',
 'timestamp': '2026-05-11T18:14:04.945513',
 'task': 'Write a Python function to count vowels in a string',
 'output': 'Okay, this sounds like a great task! I\'m ready to learn and help.\n\nHere\'s a Python function that counts vowels in a string, incorporating lessons from past experiences like handling edge cases (case-insensitivity, ignoring non-alphabetic characters), including a docstring, and providing clear example usage:\n\n```python\ndef count_vowels(s: str) -> int:\n    """\n    Counts the number of vowels (a, e, i, o, u) in a given string.\n\n    This function treats both uppercase and lowercase vowels as valid.\n    Non-alphabetic characters are ignored.\n\n    Args:\n        s: The input string to count vowels from.\n\n    Returns:\n        The total count of vowels in the string.\n    """\n',
 'score': 0.88,
 'lesson': 'Correct solution. Add type hints and handle uppercase vowels explicitly.'}

In [ ]:
def auto_score(task: str, output: str) -> tuple:
    """Ask Gemini to evaluate the quality of its own output."""

    judge_prompt = f"""You are an AI quality evaluator.

Task given to AI:
{task}

AI's output:
{output}

Evaluate this output. Respond in this exact JSON format only, no extra text, no markdown:
{{"score": 0.85, "lesson": "What the AI should do differently or better next time"}}"""

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=judge_prompt,
            config=types.GenerateContentConfig(
                temperature=0.2,        # low temp for consistent JSON
                max_output_tokens=200,
            )
        )

        # Safely get response text
        if response.text is None:
            print("[Auto-score] Empty response from Gemini, using default score.")
            return 0.75, "Could not auto-score — response was empty."

        raw = response.text.strip()

        # Strip markdown fences if present
        if "```" in raw:
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        raw = raw.strip()

        result = json.loads(raw)
        score  = float(result["score"])
        lesson = result["lesson"]
        print(f"[Auto-score] Score: {score}")
        print(f"[Auto-score] Lesson: {lesson}")
        return score, lesson

    except json.JSONDecodeError:
        print(f"[Auto-score] Could not parse JSON, raw response was:\n{raw}")
        return 0.75, "Could not auto-score — JSON parse failed."
    except Exception as e:
        print(f"[Auto-score] Error: {e}")
        return 0.75, "Could not auto-score — unexpected error."


# Run it
auto_task   = "Write a Python function that finds the largest number in a list"
auto_output = run_task(auto_task)

score, lesson = auto_score(auto_task, auto_output)

log_task(
    task   = auto_task,
    output = auto_output,
    score  = score,
    lesson = lesson,
)

print("\nFully automated task logged!")


Task: Write a Python function that finds the largest number in a list
[Memory] Found 3 relevant past example(s)
Task: Write a Python function that finds the largest number in a list
[Memory] Found 3 relevant past example(s)

Output:
Okay, this sounds like a great task! I'm ready to learn and help.

Here's a Python function that finds the largest number in a list, incorporating
[Auto-score] Could not parse JSON, raw response was:
{"score": 0.0,
[Memory] Saved task #5 (score=0.75)

Fully automated task logged!


In [ ]:
# ── CELL 14: Filter high-quality memories ────────────────────

def get_best_memories(min_score: float = 0.85) -> list:
    """Return only memories above a quality threshold."""
    memory = load_memory()
    best = [m for m in memory if m["score"] >= min_score]
    print(f"Found {len(best)} high-quality memories (score >= {min_score})")
    return best

best = get_best_memories(min_score=0.85)
for m in best:
    print(f"  [{m['score']}] {m['task'][:60]}")

Found 3 high-quality memories (score >= 0.85)
  [0.85] Write a Python function that reverses a string
  [0.9] Write a Python function that checks if a string is a palindr
  [0.88] Write a Python function to count vowels in a string


In [ ]:
def export_for_embedding():
    memory = load_memory()

    # Save locally since Drive isn't mounted
    export_path = "/content/memory_export.json"

    export = []
    for entry in memory:
        export.append({
            "id":     entry["id"],
            "text":   entry["task"],
            "output": entry["output"],
            "score":  entry["score"],
            "lesson": entry.get("lesson", ""),
        })

    with open(export_path, "w") as f:
        json.dump(export, f, indent=2)

    print(f"Exported {len(export)} entries to {export_path}")

    # Auto-download the file so you don't lose it when session ends
    from google.colab import files
    files.download(export_path)
    print("Download started — check your Downloads folder!")
    print("Save this file — you'll upload it at the start of Notebook 2.")

export_for_embedding()

Exported 5 entries to /content/memory_export.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started — check your Downloads folder!
Save this file — you'll upload it at the start of Notebook 2.
